**Reddit**

In [1]:
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW # Corrected: Import AdamW from torch.optim
from sklearn.metrics import accuracy_score, f1_score # Added f1_score
from tqdm.auto import tqdm
from safetensors.torch import save_file


# ---------- Dataset Definition ----------
class OpinionDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.df = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = str(self.df.iloc[idx]['MAIN'])
        level1 = self.df.iloc[idx]['Level 1']
        level2 = self.df.iloc[idx]['Level 2']
        level3 = self.df.iloc[idx]['Level 3']

        encoded = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        # Handle blank values as -1 for loss calculation where they are not applicable
        return {
            'input_ids': encoded['input_ids'].squeeze(),
            'attention_mask': encoded['attention_mask'].squeeze(),
            'level1': torch.tensor(level1, dtype=torch.long),
            'level2': torch.tensor(level2 if pd.notna(level2) else -1, dtype=torch.long),
            'level3': torch.tensor(level3 if pd.notna(level3) else -1, dtype=torch.long)
        }


# ---------- Model Definition ----------
class HierarchicalClassifier(nn.Module):
    def __init__(self, model_name="SarkerLab/SocBERT-base", hidden_size=768):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier1 = nn.Linear(hidden_size, 3)  # NOISE, OBJECTIVE, SUBJECTIVE
        self.classifier2 = nn.Linear(hidden_size, 3)  # NEUTRAL, NEGATIVE, POSITIVE (within SUBJECTIVE)
        self.classifier3 = nn.Linear(hidden_size, 4)  # NEUTRAL SENTIMENTS, QUESTIONS, ADVERTISEMENTS, MISCELLANEOUS (within NEUTRAL)

    def forward(self, input_ids, attention_mask, level1_labels=None, level2_labels=None, level3_labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(outputs.last_hidden_state[:, 0])

        logits1 = self.classifier1(pooled)
        logits2 = self.classifier2(pooled)
        logits3 = self.classifier3(pooled)

        loss_fct = nn.CrossEntropyLoss(ignore_index=-1, reduction='mean') # ignore_index for -1 labels
        total_loss = 0.0

        if level1_labels is not None:
            loss1 = loss_fct(logits1, level1_labels)
            total_loss += loss1

            # Only calculate Level 2 loss for subjective posts (level1 == 2)
            subjective_mask = (level1_labels == 2)
            if subjective_mask.sum() > 0 and level2_labels is not None:
                # Ensure level2_labels for non-subjective posts are ignored by CrossEntropyLoss
                # We can filter the logits and labels for subjective_mask
                # Only use valid level2_labels for loss calculation
                valid_l2_labels = level2_labels[subjective_mask]
                valid_logits2 = logits2[subjective_mask]
                
                # Filter out -1 labels from valid_l2_labels and corresponding logits
                non_ignored_l2_mask = (valid_l2_labels != -1)
                if non_ignored_l2_mask.sum() > 0:
                    loss2 = loss_fct(valid_logits2[non_ignored_l2_mask], valid_l2_labels[non_ignored_l2_mask])
                    total_loss += loss2

                    # Only calculate Level 3 loss for neutral subjective posts (level1 == 2 and level2 == 0)
                    neutral_subjective_mask = (level1_labels == 2) & (level2_labels == 0)
                    if neutral_subjective_mask.sum() > 0 and level3_labels is not None:
                        # Ensure level3_labels for non-neutral subjective posts are ignored
                        # Filter out -1 labels from neutral_subjective_mask for level3_labels
                        valid_l3_labels = level3_labels[neutral_subjective_mask]
                        valid_logits3 = logits3[neutral_subjective_mask]

                        non_ignored_l3_mask = (valid_l3_labels != -1)
                        if non_ignored_l3_mask.sum() > 0:
                            loss3 = loss_fct(valid_logits3[non_ignored_l3_mask], valid_l3_labels[non_ignored_l3_mask])
                            total_loss += loss3

        return total_loss, logits1, logits2, logits3

# ---------- Evaluation Function ----------
@torch.no_grad()
def evaluate(model, dataloader, device='cuda'):
    model.eval()
    model.to(device)

    all_l1_preds, all_l1_labels = [], []
    all_l2_preds, all_l2_labels = [], []
    all_l3_preds, all_l3_labels = [], []

    for batch in tqdm(dataloader, desc="Evaluating"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        l1 = batch['level1'].to(device)
        l2 = batch['level2'].to(device)
        l3 = batch['level3'].to(device)

        _, logits1, logits2, logits3 = model(input_ids, attention_mask)

        preds1 = torch.argmax(logits1, dim=1)
        all_l1_preds.extend(preds1.cpu().tolist())
        all_l1_labels.extend(l1.cpu().tolist())

        # Collect Level 2 predictions and labels for subjective posts (l1 == 2) that are not -1
        subjective_mask_true_l1 = (l1 == 2)
        if subjective_mask_true_l1.sum() > 0:
            preds2_filtered_raw = torch.argmax(logits2[subjective_mask_true_l1], dim=1)
            l2_filtered_true = l2[subjective_mask_true_l1]
            
            valid_l2_mask = (l2_filtered_true != -1)
            if valid_l2_mask.sum() > 0:
                all_l2_preds.extend(preds2_filtered_raw[valid_l2_mask].cpu().tolist())
                all_l2_labels.extend(l2_filtered_true[valid_l2_mask].cpu().tolist())

                # Collect Level 3 predictions and labels for neutral subjective posts (l1 == 2, l2 == 0) that are not -1
                neutral_subjective_mask_true_l2 = (l1 == 2) & (l2 == 0)
                if neutral_subjective_mask_true_l2.sum() > 0:
                    preds3_filtered_raw = torch.argmax(logits3[neutral_subjective_mask_true_l2], dim=1)
                    l3_filtered_true = l3[neutral_subjective_mask_true_l2]

                    valid_l3_mask = (l3_filtered_true != -1)
                    if valid_l3_mask.sum() > 0:
                        all_l3_preds.extend(preds3_filtered_raw[valid_l3_mask].cpu().tolist())
                        all_l3_labels.extend(l3_filtered_true[valid_l3_mask].cpu().tolist())

    l1_acc = accuracy_score(all_l1_labels, all_l1_preds)
    l1_f1 = f1_score(all_l1_labels, all_l1_preds, average='macro', zero_division=0)

    l2_acc = accuracy_score(all_l2_labels, all_l2_preds) if all_l2_labels else 0
    l2_f1 = f1_score(all_l2_labels, all_l2_preds, average='macro', zero_division=0) if all_l2_labels else 0

    l3_acc = accuracy_score(all_l3_labels, all_l3_preds) if all_l3_labels else 0
    l3_f1 = f1_score(all_l3_labels, all_l3_preds, average='macro', zero_division=0) if all_l3_labels else 0

    print(f"✅ Validation - Level 1: Acc={l1_acc:.4f}, F1={l1_f1:.4f}")
    print(f"✅ Validation - Level 2: Acc={l2_acc:.4f}, F1={l2_f1:.4f}")
    print(f"✅ Validation - Level 3: Acc={l3_acc:.4f}, F1={l3_f1:.4f}")
    
    return l1_acc, l1_f1, l2_acc, l2_f1, l3_acc, l3_f1


# ---------- Training Function ----------
def train(model, train_loader, optimizer, epochs=3, start_epoch=0, save_dir='saved_models_reddit', device='cuda', val_loader=None):
    model.to(device)
    os.makedirs(save_dir, exist_ok=True)

    best_val_combined_score = -1.0 # Initialize with a low score
    
    for epoch in range(start_epoch, epochs):
        model.train()
        total_loss = 0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            l1 = batch['level1'].to(device)
            l2 = batch['level2'].to(device)
            l3 = batch['level3'].to(device)

            optimizer.zero_grad()
            loss, _, _, _ = model(input_ids, attention_mask, l1, l2, l3)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"🟢 Epoch {epoch+1} Training Loss: {avg_loss:.4f}")

        if val_loader:
            print(f"🧪 Running validation after epoch {epoch+1}")
            l1_acc, l1_f1, l2_acc, l2_f1, l3_acc, l3_f1 = evaluate(model, val_loader, device=device)
            
            # Define a combined metric for saving the best model
            # This is a simple sum; you might want to adjust weights based on importance
            current_val_combined_score = l1_acc + l1_f1 
            
            if current_val_combined_score > best_val_combined_score:
                best_val_combined_score = current_val_combined_score
                print(f"🌟 New best validation combined score: {best_val_combined_score:.4f}. Saving model...")
                model_path = os.path.join(save_dir, "best_reddit_tc_bert_hierarchical.safetensors")
                optimizer_path = os.path.join(save_dir, "best_optimizer.pt")
                epoch_file = os.path.join(save_dir, "best_epoch.txt") # Save best epoch info

                save_file(model.state_dict(), model_path)
                torch.save(optimizer.state_dict(), optimizer_path)
                with open(epoch_file, "w") as f:
                    f.write(str(epoch+1))
                print(f"📦 Best model checkpoint saved for epoch {epoch+1}")
            else:
                print(f"No improvement. Current combined score: {current_val_combined_score:.4f}, Best: {best_val_combined_score:.4f}")

        # Always save checkpoint for resuming training (optional, but good for long runs)
        # However, for 'best model' logic, only save if it's indeed the best.
        # If you want to save every epoch's model alongside the 'best' one:
        # model_path_epoch = os.path.join(save_dir, f"reddit_tc_bert_hierarchical_epoch{epoch+1}.safetensors")
        # torch.save(model.state_dict(), model_path_epoch)

# ---------- Test Function ----------
@torch.no_grad()
def test_model(model, test_loader, device='cuda'):
    model.eval()
    model.to(device)

    all_l1_preds, all_l1_labels = [], []
    all_l2_preds, all_l2_labels = [], []
    all_l3_preds, all_l3_labels = [], []

    print("\n🚀 Starting evaluation on test data...")
    for batch in tqdm(test_loader, desc="Testing"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        l1 = batch['level1'].to(device)
        l2 = batch['level2'].to(device)
        l3 = batch['level3'].to(device)

        _, logits1, logits2, logits3 = model(input_ids, attention_mask)

        preds1 = torch.argmax(logits1, dim=1)
        all_l1_preds.extend(preds1.cpu().tolist())
        all_l1_labels.extend(l1.cpu().tolist())

        # Collect Level 2 predictions and labels for subjective posts (l1 == 2) that are not -1
        subjective_mask_true_l1 = (l1 == 2)
        if subjective_mask_true_l1.sum() > 0:
            preds2_filtered_raw = torch.argmax(logits2[subjective_mask_true_l1], dim=1)
            l2_filtered_true = l2[subjective_mask_true_l1]
            
            valid_l2_mask = (l2_filtered_true != -1)
            if valid_l2_mask.sum() > 0:
                all_l2_preds.extend(preds2_filtered_raw[valid_l2_mask].cpu().tolist())
                all_l2_labels.extend(l2_filtered_true[valid_l2_mask].cpu().tolist())

                # Collect Level 3 predictions and labels for neutral subjective posts (l1 == 2, l2 == 0) that are not -1
                neutral_subjective_mask_true_l2 = (l1 == 2) & (l2 == 0)
                if neutral_subjective_mask_true_l2.sum() > 0:
                    preds3_filtered_raw = torch.argmax(logits3[neutral_subjective_mask_true_l2], dim=1)
                    l3_filtered_true = l3[neutral_subjective_mask_true_l2]

                    valid_l3_mask = (l3_filtered_true != -1)
                    if valid_l3_mask.sum() > 0:
                        all_l3_preds.extend(preds3_filtered_raw[valid_l3_mask].cpu().tolist())
                        all_l3_labels.extend(l3_filtered_true[valid_l3_mask].cpu().tolist())

    l1_acc = accuracy_score(all_l1_labels, all_l1_preds)
    l1_f1 = f1_score(all_l1_labels, all_l1_preds, average='macro', zero_division=0)

    l2_acc = accuracy_score(all_l2_labels, all_l2_preds) if all_l2_labels else 0
    l2_f1 = f1_score(all_l2_labels, all_l2_preds, average='macro', zero_division=0) if all_l2_labels else 0

    l3_acc = accuracy_score(all_l3_labels, all_l3_preds) if all_l3_labels else 0
    l3_f1 = f1_score(all_l3_labels, all_l3_preds, average='macro', zero_division=0) if all_l3_labels else 0

    print("\n--- Test Results ---")
    print(f"🎯 Test - Level 1: Acc={l1_acc:.4f}, F1={l1_f1:.4f}")
    print(f"🎯 Test - Level 2: Acc={l2_acc:.4f}, F1={l2_f1:.4f}")
    print(f"🎯 Test - Level 3: Acc={l3_acc:.4f}, F1={l3_f1:.4f}")


# ---------- Main Execution ----------
if __name__ == "__main__":
    # Ensure you have 'train.csv' and 'test.csv' with 'text', 'Level 1', 'Level 2', 'Level 3' columns
    # Adjust paths as needed
    train_df = pd.read_csv("/kaggle/input/train-task1/reddit_train.csv")
    test_df = pd.read_csv("/kaggle/input/train-task1/reddit_test.csv")

    MODEL_NAME = "SarkerLab/SocBERT-base"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

    # Create datasets and dataloaders
    train_dataset_full = OpinionDataset(train_df, tokenizer)
    test_dataset = OpinionDataset(test_df, tokenizer)

    # Split train_dataset_full into training and validation sets
    val_size = int(0.1 * len(train_dataset_full))
    train_size = len(train_dataset_full) - val_size
    train_dataset, val_dataset = random_split(train_dataset_full, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False) # DataLoader for test data

    model = HierarchicalClassifier(model_name=MODEL_NAME)
    optimizer = AdamW(model.parameters(), lr=2e-5)

    save_dir = "saved_models_reddit"
    os.makedirs(save_dir, exist_ok=True) # Ensure save_dir exists for checking best_epoch.txt
    
    # Paths for 'best' model
    best_model_path_safetensors = os.path.join(save_dir, "best_reddit_tc_bert_hierarchical.safetensors")
    best_optimizer_path = os.path.join(save_dir, "best_optimizer.pt")
    best_epoch_file = os.path.join(save_dir, "best_epoch.txt")
    
    start_epoch = 0

    # Resume training from the LAST saved checkpoint, not necessarily the 'best'
    # For resuming, it's typically from the most recent, and then the 'best' logic kicks in
    last_epoch_file = os.path.join(save_dir, "last_epoch.txt") # This will store the last completed epoch
    if os.path.exists(last_epoch_file):
        with open(last_epoch_file, "r") as f:
            start_epoch = int(f.read())
        if start_epoch > 0:
            # Load the last checkpoint to resume
            model_to_load_path = os.path.join(save_dir, f"reddit_tc_bert_hierarchical_epoch{start_epoch}.safetensors")
            optimizer_to_load_path = os.path.join(save_dir, f"optimizer_epoch{start_epoch}.pt")

            if os.path.exists(model_to_load_path):
                print(f"Resuming training from Epoch {start_epoch} (last checkpoint).")
                from safetensors import safe_open
                state_dict = {}
                with safe_open(model_to_load_path, framework="pt", device="cpu") as f:
                    for k in f.keys():
                        state_dict[k] = f.get_tensor(k)
                model.load_state_dict(state_dict)
                
                if os.path.exists(optimizer_to_load_path):
                    optimizer.load_state_dict(torch.load(optimizer_to_load_path))
            else:
                print(f"Warning: Last checkpoint model not found at {model_to_load_path}. Starting from scratch.")
                start_epoch = 0 # Reset if checkpoint is missing


    # Train the model
    train(model, train_loader, optimizer, epochs=5, start_epoch=start_epoch, val_loader=val_loader)

    # After training, load the BEST saved model for testing
    print("\nAttempting to load the BEST saved model for final testing...")
    if os.path.exists(best_model_path_safetensors):
        try:
            from safetensors import safe_open
            state_dict = {}
            with safe_open(best_model_path_safetensors, framework="pt", device="cpu") as f:
                for k in f.keys():
                    state_dict[k] = f.get_tensor(k)
            model.load_state_dict(state_dict)
            print("Successfully loaded the best model.")
            test_model(model, test_loader)
        except Exception as e:
            print(f"Error loading best model from safetensors: {e}")
            print("Testing with the model from the last epoch of training instead.")
            test_model(model, test_loader) # Test with whatever model is currently loaded
    else:
        print("No 'best' model found. Testing with the model from the last epoch of training.")
        test_model(model, test_loader) # Test with whatever model is currently loaded

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/train-task1/reddit_train.csv'

In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm
# No need for AdamW, sklearn.metrics, random_split, save_file for inference here

# Re-define the Dataset and Model classes if they are not already defined in your current session
# (Copy-pasting them here ensures the code is self-contained and runnable)

# ---------- Dataset Definition (for inference) ----------
class InferenceDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.df = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = str(self.df.iloc[idx]['MAIN']) # Input column is 'MAIN'
        
        encoded = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            'input_ids': encoded['input_ids'].squeeze(),
            'attention_mask': encoded['attention_mask'].squeeze()
        }

# ---------- Model Definition (must be the same as trained) ----------
class HierarchicalClassifier(nn.Module):
    def __init__(self, model_name="SarkerLab/SocBERT-base", hidden_size=768):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier1 = nn.Linear(hidden_size, 3)  # NOISE, OBJECTIVE, SUBJECTIVE
        self.classifier2 = nn.Linear(hidden_size, 3)  # NEUTRAL, NEGATIVE, POSITIVE (within SUBJECTIVE)
        self.classifier3 = nn.Linear(hidden_size, 4)  # NEUTRAL SENTIMENTS, QUESTIONS, ADVERTISEMENTS, MISCELLANEOUS (within NEUTRAL)

    def forward(self, input_ids, attention_mask, level1_labels=None, level2_labels=None, level3_labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(outputs.last_hidden_state[:, 0])

        logits1 = self.classifier1(pooled)
        logits2 = self.classifier2(pooled)
        logits3 = self.classifier3(pooled)

        # For inference, we only return the logits
        return None, logits1, logits2, logits3


# --- Inference Function ---
@torch.no_grad() # Crucial for inference: disables gradient calculation
def predict_on_new_data(model, dataloader, device='cuda'):
    model.eval() # Set model to evaluation mode
    model.to(device)

    all_level1_preds = []
    all_level2_preds = []
    all_level3_preds = []

    print("\n🔮 Starting predictions on new data...")
    for batch in tqdm(dataloader, desc="Predicting"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        _, logits1, logits2, logits3 = model(input_ids, attention_mask)

        # Level 1 Prediction
        preds1 = torch.argmax(logits1, dim=1).cpu().tolist()
        all_level1_preds.extend(preds1)

        # Initialize Level 2 and Level 3 predictions as None/NaN or a placeholder
        # We'll fill them conditionally later
        current_batch_l2_preds = []
        current_batch_l3_preds = []

        # Process Level 2 and Level 3 predictions based on Level 1
        for i, l1_pred in enumerate(preds1):
            if l1_pred == 2:  # If Level 1 is SUBJECTIVE
                l2_pred = torch.argmax(logits2[i:i+1], dim=1).item()
                current_batch_l2_preds.append(l2_pred)
                
                if l2_pred == 0:  # If Level 2 is NEUTRAL
                    l3_pred = torch.argmax(logits3[i:i+1], dim=1).item()
                    current_batch_l3_preds.append(l3_pred)
                else:
                    current_batch_l3_preds.append(None) # Not applicable
            else: # If Level 1 is NOISE (0) or OBJECTIVE (1)
                current_batch_l2_preds.append(None) # Not applicable
                current_batch_l3_preds.append(None) # Not applicable
        
        all_level2_preds.extend(current_batch_l2_preds)
        all_level3_preds.extend(current_batch_l3_preds)

    return all_level1_preds, all_level2_preds, all_level3_preds


# --- Main Inference Execution ---
if __name__ == "__main__":
    # Ensure the model and tokenizer objects from your training run are still active.
    # If not, you would need to load the model state dict here:
    # MODEL_NAME = "SarkerLab/SocBERT-base"
    # tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    # model = HierarchicalClassifier(model_name=MODEL_NAME)
    # # Load the state_dict from your best saved model:
    # model_path_safetensors = os.path.join("saved_models", "best_reddit_tc_bert_hierarchical.safetensors")
    # if os.path.exists(model_path_safetensors):
    #     from safetensors import safe_open
    #     state_dict = {}
    #     with safe_open(model_path_safetensors, framework="pt", device="cpu") as f:
    #         for k in f.keys():
    #             state_dict[k] = f.get_tensor(k)
    #     model.load_state_dict(state_dict)
    #     print("Model loaded successfully for inference!")
    # else:
    #     print(f"Warning: Best model not found at {model_path_safetensors}. Ensure it was saved correctly or load another checkpoint.")
    #     # Exit or handle the error appropriately if the model cannot be loaded

    # Assume `model` and `tokenizer` are already loaded/trained from the previous script execution
    # and `device` is correctly set.

    if 'model' not in locals() or 'tokenizer' not in locals():
        print("Model or Tokenizer not found in the current environment. Please ensure the training script ran successfully or explicitly load them.")
        # As a fallback, you could add the loading logic here similar to the commented block above
        # for a fresh start or if the kernel reset.
        exit() # Exit if model/tokenizer are not available

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Load the new data for inference
    test_file_path = "/kaggle/input/train-task1/CRYPTO_REDDIT_TEST.csv"
    if not os.path.exists(test_file_path):
        print(f"Error: Test file not found at {test_file_path}. Please check the path.")
        exit()

    inference_df = pd.read_csv(test_file_path)

    # Create the inference dataset and dataloader
    inference_dataset = InferenceDataset(inference_df, tokenizer, max_len=128)
    inference_loader = DataLoader(inference_dataset, batch_size=16, shuffle=False)

    # Perform predictions
    level1_predictions, level2_predictions, level3_predictions = predict_on_new_data(model, inference_loader, device)

    # Add predictions to the DataFrame
    inference_df['level 1'] = level1_predictions
    inference_df['level 2'] = level2_predictions
    inference_df['level 3'] = level3_predictions
    
    # Handle NaN for levels where prediction is not applicable
    inference_df['level 2'] = inference_df['level 2'].fillna('')
    inference_df['level 3'] = inference_df['level 3'].fillna('')

    # Save the output file
    output_file_name = "reddit-tc-bert-crypto_test_reddit.csv"
    inference_df.to_csv(output_file_name, index=False)
    
    print(f"\nInference complete! Predictions saved to '{output_file_name}'")
    print("First 5 rows of the output file:")
    print(inference_df.head())

**Twitter**

In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW # Corrected: Import AdamW from torch.optim
from sklearn.metrics import accuracy_score, f1_score # Added f1_score
from tqdm.auto import tqdm
from safetensors.torch import save_file


# ---------- Dataset Definition ----------
class OpinionDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=64):
        self.df = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = str(self.df.iloc[idx]['Tweet'])
        level1 = self.df.iloc[idx]['Level 1']
        level2 = self.df.iloc[idx]['Level 2']
        level3 = self.df.iloc[idx]['Level 3']

        encoded = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        # Handle blank values as -1 for loss calculation where they are not applicable
        return {
            'input_ids': encoded['input_ids'].squeeze(),
            'attention_mask': encoded['attention_mask'].squeeze(),
            'level1': torch.tensor(level1, dtype=torch.long),
            'level2': torch.tensor(level2 if pd.notna(level2) else -1, dtype=torch.long),
            'level3': torch.tensor(level3 if pd.notna(level3) else -1, dtype=torch.long)
        }


# ---------- Model Definition ----------
class HierarchicalClassifier(nn.Module):
    def __init__(self, model_name="SarkerLab/SocBERT-base", hidden_size=768):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier1 = nn.Linear(hidden_size, 3)  # NOISE, OBJECTIVE, SUBJECTIVE
        self.classifier2 = nn.Linear(hidden_size, 3)  # NEUTRAL, NEGATIVE, POSITIVE (within SUBJECTIVE)
        self.classifier3 = nn.Linear(hidden_size, 4)  # NEUTRAL SENTIMENTS, QUESTIONS, ADVERTISEMENTS, MISCELLANEOUS (within NEUTRAL)

    def forward(self, input_ids, attention_mask, level1_labels=None, level2_labels=None, level3_labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(outputs.last_hidden_state[:, 0])

        logits1 = self.classifier1(pooled)
        logits2 = self.classifier2(pooled)
        logits3 = self.classifier3(pooled)

        loss_fct = nn.CrossEntropyLoss(ignore_index=-1, reduction='mean') # ignore_index for -1 labels
        total_loss = 0.0

        if level1_labels is not None:
            loss1 = loss_fct(logits1, level1_labels)
            total_loss += loss1

            # Only calculate Level 2 loss for subjective posts (level1 == 2)
            subjective_mask = (level1_labels == 2)
            if subjective_mask.sum() > 0 and level2_labels is not None:
                # Ensure level2_labels for non-subjective posts are ignored by CrossEntropyLoss
                # We can filter the logits and labels for subjective_mask
                # Only use valid level2_labels for loss calculation
                valid_l2_labels = level2_labels[subjective_mask]
                valid_logits2 = logits2[subjective_mask]
                
                # Filter out -1 labels from valid_l2_labels and corresponding logits
                non_ignored_l2_mask = (valid_l2_labels != -1)
                if non_ignored_l2_mask.sum() > 0:
                    loss2 = loss_fct(valid_logits2[non_ignored_l2_mask], valid_l2_labels[non_ignored_l2_mask])
                    total_loss += loss2

                    # Only calculate Level 3 loss for neutral subjective posts (level1 == 2 and level2 == 0)
                    neutral_subjective_mask = (level1_labels == 2) & (level2_labels == 0)
                    if neutral_subjective_mask.sum() > 0 and level3_labels is not None:
                        # Ensure level3_labels for non-neutral subjective posts are ignored
                        # Filter out -1 labels from neutral_subjective_mask for level3_labels
                        valid_l3_labels = level3_labels[neutral_subjective_mask]
                        valid_logits3 = logits3[neutral_subjective_mask]

                        non_ignored_l3_mask = (valid_l3_labels != -1)
                        if non_ignored_l3_mask.sum() > 0:
                            loss3 = loss_fct(valid_logits3[non_ignored_l3_mask], valid_l3_labels[non_ignored_l3_mask])
                            total_loss += loss3

        return total_loss, logits1, logits2, logits3

# ---------- Evaluation Function ----------
@torch.no_grad()
def evaluate(model, dataloader, device='cuda'):
    model.eval()
    model.to(device)

    all_l1_preds, all_l1_labels = [], []
    all_l2_preds, all_l2_labels = [], []
    all_l3_preds, all_l3_labels = [], []

    for batch in tqdm(dataloader, desc="Evaluating"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        l1 = batch['level1'].to(device)
        l2 = batch['level2'].to(device)
        l3 = batch['level3'].to(device)

        _, logits1, logits2, logits3 = model(input_ids, attention_mask)

        preds1 = torch.argmax(logits1, dim=1)
        all_l1_preds.extend(preds1.cpu().tolist())
        all_l1_labels.extend(l1.cpu().tolist())

        # Collect Level 2 predictions and labels for subjective posts (l1 == 2) that are not -1
        subjective_mask_true_l1 = (l1 == 2)
        if subjective_mask_true_l1.sum() > 0:
            preds2_filtered_raw = torch.argmax(logits2[subjective_mask_true_l1], dim=1)
            l2_filtered_true = l2[subjective_mask_true_l1]
            
            valid_l2_mask = (l2_filtered_true != -1)
            if valid_l2_mask.sum() > 0:
                all_l2_preds.extend(preds2_filtered_raw[valid_l2_mask].cpu().tolist())
                all_l2_labels.extend(l2_filtered_true[valid_l2_mask].cpu().tolist())

                # Collect Level 3 predictions and labels for neutral subjective posts (l1 == 2, l2 == 0) that are not -1
                neutral_subjective_mask_true_l2 = (l1 == 2) & (l2 == 0)
                if neutral_subjective_mask_true_l2.sum() > 0:
                    preds3_filtered_raw = torch.argmax(logits3[neutral_subjective_mask_true_l2], dim=1)
                    l3_filtered_true = l3[neutral_subjective_mask_true_l2]

                    valid_l3_mask = (l3_filtered_true != -1)
                    if valid_l3_mask.sum() > 0:
                        all_l3_preds.extend(preds3_filtered_raw[valid_l3_mask].cpu().tolist())
                        all_l3_labels.extend(l3_filtered_true[valid_l3_mask].cpu().tolist())

    l1_acc = accuracy_score(all_l1_labels, all_l1_preds)
    l1_f1 = f1_score(all_l1_labels, all_l1_preds, average='macro', zero_division=0)

    l2_acc = accuracy_score(all_l2_labels, all_l2_preds) if all_l2_labels else 0
    l2_f1 = f1_score(all_l2_labels, all_l2_preds, average='macro', zero_division=0) if all_l2_labels else 0

    l3_acc = accuracy_score(all_l3_labels, all_l3_preds) if all_l3_labels else 0
    l3_f1 = f1_score(all_l3_labels, all_l3_preds, average='macro', zero_division=0) if all_l3_labels else 0

    print(f"✅ Validation - Level 1: Acc={l1_acc:.4f}, F1={l1_f1:.4f}")
    print(f"✅ Validation - Level 2: Acc={l2_acc:.4f}, F1={l2_f1:.4f}")
    print(f"✅ Validation - Level 3: Acc={l3_acc:.4f}, F1={l3_f1:.4f}")
    
    return l1_acc, l1_f1, l2_acc, l2_f1, l3_acc, l3_f1


# ---------- Training Function ----------
def train(model, train_loader, optimizer, epochs=3, start_epoch=0, save_dir='saved_models_twitter', device='cuda', val_loader=None):
    model.to(device)
    os.makedirs(save_dir, exist_ok=True)

    best_val_combined_score = -1.0 # Initialize with a low score
    
    for epoch in range(start_epoch, epochs):
        model.train()
        total_loss = 0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            l1 = batch['level1'].to(device)
            l2 = batch['level2'].to(device)
            l3 = batch['level3'].to(device)

            optimizer.zero_grad()
            loss, _, _, _ = model(input_ids, attention_mask, l1, l2, l3)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"🟢 Epoch {epoch+1} Training Loss: {avg_loss:.4f}")

        if val_loader:
            print(f"🧪 Running validation after epoch {epoch+1}")
            l1_acc, l1_f1, l2_acc, l2_f1, l3_acc, l3_f1 = evaluate(model, val_loader, device=device)
            
            # Define a combined metric for saving the best model
            # This is a simple sum; you might want to adjust weights based on importance
            current_val_combined_score = l1_acc + l1_f1 
            
            if current_val_combined_score > best_val_combined_score:
                best_val_combined_score = current_val_combined_score
                print(f"🌟 New best validation combined score: {best_val_combined_score:.4f}. Saving model...")
                model_path = os.path.join(save_dir, "best_reddit_tc_bert_hierarchical.safetensors")
                optimizer_path = os.path.join(save_dir, "best_optimizer.pt")
                epoch_file = os.path.join(save_dir, "best_epoch.txt") # Save best epoch info

                save_file(model.state_dict(), model_path)
                torch.save(optimizer.state_dict(), optimizer_path)
                with open(epoch_file, "w") as f:
                    f.write(str(epoch+1))
                print(f"📦 Best model checkpoint saved for epoch {epoch+1}")
            else:
                print(f"No improvement. Current combined score: {current_val_combined_score:.4f}, Best: {best_val_combined_score:.4f}")

        # Always save checkpoint for resuming training (optional, but good for long runs)
        # However, for 'best model' logic, only save if it's indeed the best.
        # If you want to save every epoch's model alongside the 'best' one:
        # model_path_epoch = os.path.join(save_dir, f"reddit_tc_bert_hierarchical_epoch{epoch+1}.safetensors")
        # torch.save(model.state_dict(), model_path_epoch)

# ---------- Test Function ----------
@torch.no_grad()
def test_model(model, test_loader, device='cuda'):
    model.eval()
    model.to(device)

    all_l1_preds, all_l1_labels = [], []
    all_l2_preds, all_l2_labels = [], []
    all_l3_preds, all_l3_labels = [], []

    print("\n🚀 Starting evaluation on test data...")
    for batch in tqdm(test_loader, desc="Testing"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        l1 = batch['level1'].to(device)
        l2 = batch['level2'].to(device)
        l3 = batch['level3'].to(device)

        _, logits1, logits2, logits3 = model(input_ids, attention_mask)

        preds1 = torch.argmax(logits1, dim=1)
        all_l1_preds.extend(preds1.cpu().tolist())
        all_l1_labels.extend(l1.cpu().tolist())

        # Collect Level 2 predictions and labels for subjective posts (l1 == 2) that are not -1
        subjective_mask_true_l1 = (l1 == 2)
        if subjective_mask_true_l1.sum() > 0:
            preds2_filtered_raw = torch.argmax(logits2[subjective_mask_true_l1], dim=1)
            l2_filtered_true = l2[subjective_mask_true_l1]
            
            valid_l2_mask = (l2_filtered_true != -1)
            if valid_l2_mask.sum() > 0:
                all_l2_preds.extend(preds2_filtered_raw[valid_l2_mask].cpu().tolist())
                all_l2_labels.extend(l2_filtered_true[valid_l2_mask].cpu().tolist())

                # Collect Level 3 predictions and labels for neutral subjective posts (l1 == 2, l2 == 0) that are not -1
                neutral_subjective_mask_true_l2 = (l1 == 2) & (l2 == 0)
                if neutral_subjective_mask_true_l2.sum() > 0:
                    preds3_filtered_raw = torch.argmax(logits3[neutral_subjective_mask_true_l2], dim=1)
                    l3_filtered_true = l3[neutral_subjective_mask_true_l2]

                    valid_l3_mask = (l3_filtered_true != -1)
                    if valid_l3_mask.sum() > 0:
                        all_l3_preds.extend(preds3_filtered_raw[valid_l3_mask].cpu().tolist())
                        all_l3_labels.extend(l3_filtered_true[valid_l3_mask].cpu().tolist())

    l1_acc = accuracy_score(all_l1_labels, all_l1_preds)
    l1_f1 = f1_score(all_l1_labels, all_l1_preds, average='macro', zero_division=0)

    l2_acc = accuracy_score(all_l2_labels, all_l2_preds) if all_l2_labels else 0
    l2_f1 = f1_score(all_l2_labels, all_l2_preds, average='macro', zero_division=0) if all_l2_labels else 0

    l3_acc = accuracy_score(all_l3_labels, all_l3_preds) if all_l3_labels else 0
    l3_f1 = f1_score(all_l3_labels, all_l3_preds, average='macro', zero_division=0) if all_l3_labels else 0

    print("\n--- Test Results ---")
    print(f"🎯 Test - Level 1: Acc={l1_acc:.4f}, F1={l1_f1:.4f}")
    print(f"🎯 Test - Level 2: Acc={l2_acc:.4f}, F1={l2_f1:.4f}")
    print(f"🎯 Test - Level 3: Acc={l3_acc:.4f}, F1={l3_f1:.4f}")


# ---------- Main Execution ----------
if __name__ == "__main__":
    # Ensure you have 'train.csv' and 'test.csv' with 'text', 'Level 1', 'Level 2', 'Level 3' columns
    # Adjust paths as needed
    train_df = pd.read_csv("/kaggle/input/train-task1/twitter_train.csv")
    test_df = pd.read_csv("/kaggle/input/train-task1/twitter_test.csv")

    MODEL_NAME = "SarkerLab/SocBERT-base"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

    # Create datasets and dataloaders
    train_dataset_full = OpinionDataset(train_df, tokenizer)
    test_dataset = OpinionDataset(test_df, tokenizer)

    # Split train_dataset_full into training and validation sets
    val_size = int(0.1 * len(train_dataset_full))
    train_size = len(train_dataset_full) - val_size
    train_dataset, val_dataset = random_split(train_dataset_full, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False) # DataLoader for test data

    model = HierarchicalClassifier(model_name=MODEL_NAME)
    optimizer = AdamW(model.parameters(), lr=2e-5)

    save_dir = "saved_models_twitter"
    os.makedirs(save_dir, exist_ok=True) # Ensure save_dir exists for checking best_epoch.txt
    
    # Paths for 'best' model
    best_model_path_safetensors = os.path.join(save_dir, "best_reddit_tc_bert_hierarchical.safetensors")
    best_optimizer_path = os.path.join(save_dir, "best_optimizer.pt")
    best_epoch_file = os.path.join(save_dir, "best_epoch.txt")
    
    start_epoch = 0

    # Resume training from the LAST saved checkpoint, not necessarily the 'best'
    # For resuming, it's typically from the most recent, and then the 'best' logic kicks in
    last_epoch_file = os.path.join(save_dir, "last_epoch.txt") # This will store the last completed epoch
    if os.path.exists(last_epoch_file):
        with open(last_epoch_file, "r") as f:
            start_epoch = int(f.read())
        if start_epoch > 0:
            # Load the last checkpoint to resume
            model_to_load_path = os.path.join(save_dir, f"reddit_tc_bert_hierarchical_epoch{start_epoch}.safetensors")
            optimizer_to_load_path = os.path.join(save_dir, f"optimizer_epoch{start_epoch}.pt")

            if os.path.exists(model_to_load_path):
                print(f"Resuming training from Epoch {start_epoch} (last checkpoint).")
                from safetensors import safe_open
                state_dict = {}
                with safe_open(model_to_load_path, framework="pt", device="cpu") as f:
                    for k in f.keys():
                        state_dict[k] = f.get_tensor(k)
                model.load_state_dict(state_dict)
                
                if os.path.exists(optimizer_to_load_path):
                    optimizer.load_state_dict(torch.load(optimizer_to_load_path))
            else:
                print(f"Warning: Last checkpoint model not found at {model_to_load_path}. Starting from scratch.")
                start_epoch = 0 # Reset if checkpoint is missing


    # Train the model
    train(model, train_loader, optimizer, epochs=7, start_epoch=start_epoch, val_loader=val_loader)

    # After training, load the BEST saved model for testing
    print("\nAttempting to load the BEST saved model for final testing...")
    if os.path.exists(best_model_path_safetensors):
        try:
            from safetensors import safe_open
            state_dict = {}
            with safe_open(best_model_path_safetensors, framework="pt", device="cpu") as f:
                for k in f.keys():
                    state_dict[k] = f.get_tensor(k)
            model.load_state_dict(state_dict)
            print("Successfully loaded the best model.")
            test_model(model, test_loader)
        except Exception as e:
            print(f"Error loading best model from safetensors: {e}")
            print("Testing with the model from the last epoch of training instead.")
            test_model(model, test_loader) # Test with whatever model is currently loaded
    else:
        print("No 'best' model found. Testing with the model from the last epoch of training.")
        test_model(model, test_loader) # Test with whatever model is currently loaded

In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm
# No need for AdamW, sklearn.metrics, random_split, save_file for inference here

# Re-define the Dataset and Model classes if they are not already defined in your current session
# (Copy-pasting them here ensures the code is self-contained and runnable)

# ---------- Dataset Definition (for inference) ----------
class InferenceDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.df = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = str(self.df.iloc[idx]['Text']) 
        
        encoded = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            'input_ids': encoded['input_ids'].squeeze(),
            'attention_mask': encoded['attention_mask'].squeeze()
        }

# ---------- Model Definition (must be the same as trained) ----------
class HierarchicalClassifier(nn.Module):
    def __init__(self, model_name="SarkerLab/SocBERT-base", hidden_size=768):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier1 = nn.Linear(hidden_size, 3)  # NOISE, OBJECTIVE, SUBJECTIVE
        self.classifier2 = nn.Linear(hidden_size, 3)  # NEUTRAL, NEGATIVE, POSITIVE (within SUBJECTIVE)
        self.classifier3 = nn.Linear(hidden_size, 4)  # NEUTRAL SENTIMENTS, QUESTIONS, ADVERTISEMENTS, MISCELLANEOUS (within NEUTRAL)

    def forward(self, input_ids, attention_mask, level1_labels=None, level2_labels=None, level3_labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(outputs.last_hidden_state[:, 0])

        logits1 = self.classifier1(pooled)
        logits2 = self.classifier2(pooled)
        logits3 = self.classifier3(pooled)

        # For inference, we only return the logits
        return None, logits1, logits2, logits3


# --- Inference Function ---
@torch.no_grad() # Crucial for inference: disables gradient calculation
def predict_on_new_data(model, dataloader, device='cuda'):
    model.eval() # Set model to evaluation mode
    model.to(device)

    all_level1_preds = []
    all_level2_preds = []
    all_level3_preds = []

    print("\n🔮 Starting predictions on new data...")
    for batch in tqdm(dataloader, desc="Predicting"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        _, logits1, logits2, logits3 = model(input_ids, attention_mask)

        # Level 1 Prediction
        preds1 = torch.argmax(logits1, dim=1).cpu().tolist()
        all_level1_preds.extend(preds1)

        # Initialize Level 2 and Level 3 predictions as None/NaN or a placeholder
        # We'll fill them conditionally later
        current_batch_l2_preds = []
        current_batch_l3_preds = []

        # Process Level 2 and Level 3 predictions based on Level 1
        for i, l1_pred in enumerate(preds1):
            if l1_pred == 2:  # If Level 1 is SUBJECTIVE
                l2_pred = torch.argmax(logits2[i:i+1], dim=1).item()
                current_batch_l2_preds.append(l2_pred)
                
                if l2_pred == 0:  # If Level 2 is NEUTRAL
                    l3_pred = torch.argmax(logits3[i:i+1], dim=1).item()
                    current_batch_l3_preds.append(l3_pred)
                else:
                    current_batch_l3_preds.append(None) # Not applicable
            else: # If Level 1 is NOISE (0) or OBJECTIVE (1)
                current_batch_l2_preds.append(None) # Not applicable
                current_batch_l3_preds.append(None) # Not applicable
        
        all_level2_preds.extend(current_batch_l2_preds)
        all_level3_preds.extend(current_batch_l3_preds)

    return all_level1_preds, all_level2_preds, all_level3_preds


# --- Main Inference Execution ---
if __name__ == "__main__":
    # Ensure the model and tokenizer objects from your training run are still active.
    # If not, you would need to load the model state dict here:
    # MODEL_NAME = "SarkerLab/SocBERT-base"
    # tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    # model = HierarchicalClassifier(model_name=MODEL_NAME)
    # # Load the state_dict from your best saved model:
    # model_path_safetensors = os.path.join("saved_models", "best_reddit_tc_bert_hierarchical.safetensors")
    # if os.path.exists(model_path_safetensors):
    #     from safetensors import safe_open
    #     state_dict = {}
    #     with safe_open(model_path_safetensors, framework="pt", device="cpu") as f:
    #         for k in f.keys():
    #             state_dict[k] = f.get_tensor(k)
    #     model.load_state_dict(state_dict)
    #     print("Model loaded successfully for inference!")
    # else:
    #     print(f"Warning: Best model not found at {model_path_safetensors}. Ensure it was saved correctly or load another checkpoint.")
    #     # Exit or handle the error appropriately if the model cannot be loaded

    # Assume `model` and `tokenizer` are already loaded/trained from the previous script execution
    # and `device` is correctly set.

    if 'model' not in locals() or 'tokenizer' not in locals():
        print("Model or Tokenizer not found in the current environment. Please ensure the training script ran successfully or explicitly load them.")
        # As a fallback, you could add the loading logic here similar to the commented block above
        # for a fresh start or if the kernel reset.
        exit() # Exit if model/tokenizer are not available

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Load the new data for inference
    test_file_path = "/kaggle/input/train-task1/CRYPTO_TWITTER_TEST.csv"
    if not os.path.exists(test_file_path):
        print(f"Error: Test file not found at {test_file_path}. Please check the path.")
        exit()

    inference_df = pd.read_csv(test_file_path)

    # Create the inference dataset and dataloader
    inference_dataset = InferenceDataset(inference_df, tokenizer, max_len=128)
    inference_loader = DataLoader(inference_dataset, batch_size=16, shuffle=False)

    # Perform predictions
    level1_predictions, level2_predictions, level3_predictions = predict_on_new_data(model, inference_loader, device)

    # Add predictions to the DataFrame
    inference_df['level 1'] = level1_predictions
    inference_df['level 2'] = level2_predictions
    inference_df['level 3'] = level3_predictions
    
    # Handle NaN for levels where prediction is not applicable
    inference_df['level 2'] = inference_df['level 2'].fillna('')
    inference_df['level 3'] = inference_df['level 3'].fillna('')

    # Save the output file
    output_file_name = "reddit-tc-bert-crypto_test_twitter.csv"
    inference_df.to_csv(output_file_name, index=False)
    
    print(f"\nInference complete! Predictions saved to '{output_file_name}'")
    print("First 5 rows of the output file:")
    print(inference_df.head())

**Youtube**

In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW # Corrected: Import AdamW from torch.optim
from sklearn.metrics import accuracy_score, f1_score # Added f1_score
from tqdm.auto import tqdm
from safetensors.torch import save_file


# ---------- Dataset Definition ----------
class OpinionDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.df = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = str(self.df.iloc[idx]['comment'])
        level1 = self.df.iloc[idx]['Level 1']
        level2 = self.df.iloc[idx]['Level 2']
        level3 = self.df.iloc[idx]['Level 3']

        encoded = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        # Handle blank values as -1 for loss calculation where they are not applicable
        return {
            'input_ids': encoded['input_ids'].squeeze(),
            'attention_mask': encoded['attention_mask'].squeeze(),
            'level1': torch.tensor(level1, dtype=torch.long),
            'level2': torch.tensor(level2 if pd.notna(level2) else -1, dtype=torch.long),
            'level3': torch.tensor(level3 if pd.notna(level3) else -1, dtype=torch.long)
        }


# ---------- Model Definition ----------
class HierarchicalClassifier(nn.Module):
    def __init__(self, model_name="SarkerLab/SocBERT-base", hidden_size=768):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier1 = nn.Linear(hidden_size, 3)  # NOISE, OBJECTIVE, SUBJECTIVE
        self.classifier2 = nn.Linear(hidden_size, 3)  # NEUTRAL, NEGATIVE, POSITIVE (within SUBJECTIVE)
        self.classifier3 = nn.Linear(hidden_size, 4)  # NEUTRAL SENTIMENTS, QUESTIONS, ADVERTISEMENTS, MISCELLANEOUS (within NEUTRAL)

    def forward(self, input_ids, attention_mask, level1_labels=None, level2_labels=None, level3_labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(outputs.last_hidden_state[:, 0])

        logits1 = self.classifier1(pooled)
        logits2 = self.classifier2(pooled)
        logits3 = self.classifier3(pooled)

        loss_fct = nn.CrossEntropyLoss(ignore_index=-1, reduction='mean') # ignore_index for -1 labels
        total_loss = 0.0

        if level1_labels is not None:
            loss1 = loss_fct(logits1, level1_labels)
            total_loss += loss1

            # Only calculate Level 2 loss for subjective posts (level1 == 2)
            subjective_mask = (level1_labels == 2)
            if subjective_mask.sum() > 0 and level2_labels is not None:
                # Ensure level2_labels for non-subjective posts are ignored by CrossEntropyLoss
                # We can filter the logits and labels for subjective_mask
                # Only use valid level2_labels for loss calculation
                valid_l2_labels = level2_labels[subjective_mask]
                valid_logits2 = logits2[subjective_mask]
                
                # Filter out -1 labels from valid_l2_labels and corresponding logits
                non_ignored_l2_mask = (valid_l2_labels != -1)
                if non_ignored_l2_mask.sum() > 0:
                    loss2 = loss_fct(valid_logits2[non_ignored_l2_mask], valid_l2_labels[non_ignored_l2_mask])
                    total_loss += loss2

                    # Only calculate Level 3 loss for neutral subjective posts (level1 == 2 and level2 == 0)
                    neutral_subjective_mask = (level1_labels == 2) & (level2_labels == 0)
                    if neutral_subjective_mask.sum() > 0 and level3_labels is not None:
                        # Ensure level3_labels for non-neutral subjective posts are ignored
                        # Filter out -1 labels from neutral_subjective_mask for level3_labels
                        valid_l3_labels = level3_labels[neutral_subjective_mask]
                        valid_logits3 = logits3[neutral_subjective_mask]

                        non_ignored_l3_mask = (valid_l3_labels != -1)
                        if non_ignored_l3_mask.sum() > 0:
                            loss3 = loss_fct(valid_logits3[non_ignored_l3_mask], valid_l3_labels[non_ignored_l3_mask])
                            total_loss += loss3

        return total_loss, logits1, logits2, logits3

# ---------- Evaluation Function ----------
@torch.no_grad()
def evaluate(model, dataloader, device='cuda'):
    model.eval()
    model.to(device)

    all_l1_preds, all_l1_labels = [], []
    all_l2_preds, all_l2_labels = [], []
    all_l3_preds, all_l3_labels = [], []

    for batch in tqdm(dataloader, desc="Evaluating"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        l1 = batch['level1'].to(device)
        l2 = batch['level2'].to(device)
        l3 = batch['level3'].to(device)

        _, logits1, logits2, logits3 = model(input_ids, attention_mask)

        preds1 = torch.argmax(logits1, dim=1)
        all_l1_preds.extend(preds1.cpu().tolist())
        all_l1_labels.extend(l1.cpu().tolist())

        # Collect Level 2 predictions and labels for subjective posts (l1 == 2) that are not -1
        subjective_mask_true_l1 = (l1 == 2)
        if subjective_mask_true_l1.sum() > 0:
            preds2_filtered_raw = torch.argmax(logits2[subjective_mask_true_l1], dim=1)
            l2_filtered_true = l2[subjective_mask_true_l1]
            
            valid_l2_mask = (l2_filtered_true != -1)
            if valid_l2_mask.sum() > 0:
                all_l2_preds.extend(preds2_filtered_raw[valid_l2_mask].cpu().tolist())
                all_l2_labels.extend(l2_filtered_true[valid_l2_mask].cpu().tolist())

                # Collect Level 3 predictions and labels for neutral subjective posts (l1 == 2, l2 == 0) that are not -1
                neutral_subjective_mask_true_l2 = (l1 == 2) & (l2 == 0)
                if neutral_subjective_mask_true_l2.sum() > 0:
                    preds3_filtered_raw = torch.argmax(logits3[neutral_subjective_mask_true_l2], dim=1)
                    l3_filtered_true = l3[neutral_subjective_mask_true_l2]

                    valid_l3_mask = (l3_filtered_true != -1)
                    if valid_l3_mask.sum() > 0:
                        all_l3_preds.extend(preds3_filtered_raw[valid_l3_mask].cpu().tolist())
                        all_l3_labels.extend(l3_filtered_true[valid_l3_mask].cpu().tolist())

    l1_acc = accuracy_score(all_l1_labels, all_l1_preds)
    l1_f1 = f1_score(all_l1_labels, all_l1_preds, average='macro', zero_division=0)

    l2_acc = accuracy_score(all_l2_labels, all_l2_preds) if all_l2_labels else 0
    l2_f1 = f1_score(all_l2_labels, all_l2_preds, average='macro', zero_division=0) if all_l2_labels else 0

    l3_acc = accuracy_score(all_l3_labels, all_l3_preds) if all_l3_labels else 0
    l3_f1 = f1_score(all_l3_labels, all_l3_preds, average='macro', zero_division=0) if all_l3_labels else 0

    print(f"✅ Validation - Level 1: Acc={l1_acc:.4f}, F1={l1_f1:.4f}")
    print(f"✅ Validation - Level 2: Acc={l2_acc:.4f}, F1={l2_f1:.4f}")
    print(f"✅ Validation - Level 3: Acc={l3_acc:.4f}, F1={l3_f1:.4f}")
    
    return l1_acc, l1_f1, l2_acc, l2_f1, l3_acc, l3_f1


# ---------- Training Function ----------
def train(model, train_loader, optimizer, epochs=3, start_epoch=0, save_dir='saved_models_youtube', device='cuda', val_loader=None):
    model.to(device)
    os.makedirs(save_dir, exist_ok=True)

    best_val_combined_score = -1.0 # Initialize with a low score
    
    for epoch in range(start_epoch, epochs):
        model.train()
        total_loss = 0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            l1 = batch['level1'].to(device)
            l2 = batch['level2'].to(device)
            l3 = batch['level3'].to(device)

            optimizer.zero_grad()
            loss, _, _, _ = model(input_ids, attention_mask, l1, l2, l3)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"🟢 Epoch {epoch+1} Training Loss: {avg_loss:.4f}")

        if val_loader:
            print(f"🧪 Running validation after epoch {epoch+1}")
            l1_acc, l1_f1, l2_acc, l2_f1, l3_acc, l3_f1 = evaluate(model, val_loader, device=device)
            
            # Define a combined metric for saving the best model
            # This is a simple sum; you might want to adjust weights based on importance
            current_val_combined_score = l1_acc + l1_f1 
            
            if current_val_combined_score > best_val_combined_score:
                best_val_combined_score = current_val_combined_score
                print(f"🌟 New best validation combined score: {best_val_combined_score:.4f}. Saving model...")
                model_path = os.path.join(save_dir, "best_reddit_tc_bert_hierarchical.safetensors")
                optimizer_path = os.path.join(save_dir, "best_optimizer.pt")
                epoch_file = os.path.join(save_dir, "best_epoch.txt") # Save best epoch info

                save_file(model.state_dict(), model_path)
                torch.save(optimizer.state_dict(), optimizer_path)
                with open(epoch_file, "w") as f:
                    f.write(str(epoch+1))
                print(f"📦 Best model checkpoint saved for epoch {epoch+1}")
            else:
                print(f"No improvement. Current combined score: {current_val_combined_score:.4f}, Best: {best_val_combined_score:.4f}")

        # Always save checkpoint for resuming training (optional, but good for long runs)
        # However, for 'best model' logic, only save if it's indeed the best.
        # If you want to save every epoch's model alongside the 'best' one:
        # model_path_epoch = os.path.join(save_dir, f"reddit_tc_bert_hierarchical_epoch{epoch+1}.safetensors")
        # torch.save(model.state_dict(), model_path_epoch)

# ---------- Test Function ----------
@torch.no_grad()
def test_model(model, test_loader, device='cuda'):
    model.eval()
    model.to(device)

    all_l1_preds, all_l1_labels = [], []
    all_l2_preds, all_l2_labels = [], []
    all_l3_preds, all_l3_labels = [], []

    print("\n🚀 Starting evaluation on test data...")
    for batch in tqdm(test_loader, desc="Testing"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        l1 = batch['level1'].to(device)
        l2 = batch['level2'].to(device)
        l3 = batch['level3'].to(device)

        _, logits1, logits2, logits3 = model(input_ids, attention_mask)

        preds1 = torch.argmax(logits1, dim=1)
        all_l1_preds.extend(preds1.cpu().tolist())
        all_l1_labels.extend(l1.cpu().tolist())

        # Collect Level 2 predictions and labels for subjective posts (l1 == 2) that are not -1
        subjective_mask_true_l1 = (l1 == 2)
        if subjective_mask_true_l1.sum() > 0:
            preds2_filtered_raw = torch.argmax(logits2[subjective_mask_true_l1], dim=1)
            l2_filtered_true = l2[subjective_mask_true_l1]
            
            valid_l2_mask = (l2_filtered_true != -1)
            if valid_l2_mask.sum() > 0:
                all_l2_preds.extend(preds2_filtered_raw[valid_l2_mask].cpu().tolist())
                all_l2_labels.extend(l2_filtered_true[valid_l2_mask].cpu().tolist())

                # Collect Level 3 predictions and labels for neutral subjective posts (l1 == 2, l2 == 0) that are not -1
                neutral_subjective_mask_true_l2 = (l1 == 2) & (l2 == 0)
                if neutral_subjective_mask_true_l2.sum() > 0:
                    preds3_filtered_raw = torch.argmax(logits3[neutral_subjective_mask_true_l2], dim=1)
                    l3_filtered_true = l3[neutral_subjective_mask_true_l2]

                    valid_l3_mask = (l3_filtered_true != -1)
                    if valid_l3_mask.sum() > 0:
                        all_l3_preds.extend(preds3_filtered_raw[valid_l3_mask].cpu().tolist())
                        all_l3_labels.extend(l3_filtered_true[valid_l3_mask].cpu().tolist())

    l1_acc = accuracy_score(all_l1_labels, all_l1_preds)
    l1_f1 = f1_score(all_l1_labels, all_l1_preds, average='macro', zero_division=0)

    l2_acc = accuracy_score(all_l2_labels, all_l2_preds) if all_l2_labels else 0
    l2_f1 = f1_score(all_l2_labels, all_l2_preds, average='macro', zero_division=0) if all_l2_labels else 0

    l3_acc = accuracy_score(all_l3_labels, all_l3_preds) if all_l3_labels else 0
    l3_f1 = f1_score(all_l3_labels, all_l3_preds, average='macro', zero_division=0) if all_l3_labels else 0

    print("\n--- Test Results ---")
    print(f"🎯 Test - Level 1: Acc={l1_acc:.4f}, F1={l1_f1:.4f}")
    print(f"🎯 Test - Level 2: Acc={l2_acc:.4f}, F1={l2_f1:.4f}")
    print(f"🎯 Test - Level 3: Acc={l3_acc:.4f}, F1={l3_f1:.4f}")


# ---------- Main Execution ----------
if __name__ == "__main__":
    # Ensure you have 'train.csv' and 'test.csv' with 'text', 'Level 1', 'Level 2', 'Level 3' columns
    # Adjust paths as needed
    train_df = pd.read_csv("/kaggle/input/train-task1/youtube_train.csv")
    test_df = pd.read_csv("/kaggle/input/train-task1/youtube_test.csv")

    MODEL_NAME = "SarkerLab/SocBERT-base"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

    # Create datasets and dataloaders
    train_dataset_full = OpinionDataset(train_df, tokenizer)
    test_dataset = OpinionDataset(test_df, tokenizer)

    # Split train_dataset_full into training and validation sets
    val_size = int(0.1 * len(train_dataset_full))
    train_size = len(train_dataset_full) - val_size
    train_dataset, val_dataset = random_split(train_dataset_full, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False) # DataLoader for test data

    model = HierarchicalClassifier(model_name=MODEL_NAME)
    optimizer = AdamW(model.parameters(), lr=2e-5)

    save_dir = "saved_models_youtube"
    os.makedirs(save_dir, exist_ok=True) # Ensure save_dir exists for checking best_epoch.txt
    
    # Paths for 'best' model
    best_model_path_safetensors = os.path.join(save_dir, "best_reddit_tc_bert_hierarchical.safetensors")
    best_optimizer_path = os.path.join(save_dir, "best_optimizer.pt")
    best_epoch_file = os.path.join(save_dir, "best_epoch.txt")
    
    start_epoch = 0

    # Resume training from the LAST saved checkpoint, not necessarily the 'best'
    # For resuming, it's typically from the most recent, and then the 'best' logic kicks in
    last_epoch_file = os.path.join(save_dir, "last_epoch.txt") # This will store the last completed epoch
    if os.path.exists(last_epoch_file):
        with open(last_epoch_file, "r") as f:
            start_epoch = int(f.read())
        if start_epoch > 0:
            # Load the last checkpoint to resume
            model_to_load_path = os.path.join(save_dir, f"reddit_tc_bert_hierarchical_epoch{start_epoch}.safetensors")
            optimizer_to_load_path = os.path.join(save_dir, f"optimizer_epoch{start_epoch}.pt")

            if os.path.exists(model_to_load_path):
                print(f"Resuming training from Epoch {start_epoch} (last checkpoint).")
                from safetensors import safe_open
                state_dict = {}
                with safe_open(model_to_load_path, framework="pt", device="cpu") as f:
                    for k in f.keys():
                        state_dict[k] = f.get_tensor(k)
                model.load_state_dict(state_dict)
                
                if os.path.exists(optimizer_to_load_path):
                    optimizer.load_state_dict(torch.load(optimizer_to_load_path))
            else:
                print(f"Warning: Last checkpoint model not found at {model_to_load_path}. Starting from scratch.")
                start_epoch = 0 # Reset if checkpoint is missing


    # Train the model
    train(model, train_loader, optimizer, epochs=5, start_epoch=start_epoch, val_loader=val_loader)

    # After training, load the BEST saved model for testing
    print("\nAttempting to load the BEST saved model for final testing...")
    if os.path.exists(best_model_path_safetensors):
        try:
            from safetensors import safe_open
            state_dict = {}
            with safe_open(best_model_path_safetensors, framework="pt", device="cpu") as f:
                for k in f.keys():
                    state_dict[k] = f.get_tensor(k)
            model.load_state_dict(state_dict)
            print("Successfully loaded the best model.")
            test_model(model, test_loader)
        except Exception as e:
            print(f"Error loading best model from safetensors: {e}")
            print("Testing with the model from the last epoch of training instead.")
            test_model(model, test_loader) # Test with whatever model is currently loaded
    else:
        print("No 'best' model found. Testing with the model from the last epoch of training.")
        test_model(model, test_loader) # Test with whatever model is currently loaded

In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm
# No need for AdamW, sklearn.metrics, random_split, save_file for inference here

# Re-define the Dataset and Model classes if they are not already defined in your current session
# (Copy-pasting them here ensures the code is self-contained and runnable)

# ---------- Dataset Definition (for inference) ----------
class InferenceDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=64):
        self.df = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = str(self.df.iloc[idx]['MAIN']) # Input column is 'MAIN'
        
        encoded = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            'input_ids': encoded['input_ids'].squeeze(),
            'attention_mask': encoded['attention_mask'].squeeze()
        }

# ---------- Model Definition (must be the same as trained) ----------
class HierarchicalClassifier(nn.Module):
    def __init__(self, model_name="SarkerLab/SocBERT-base", hidden_size=768):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier1 = nn.Linear(hidden_size, 3)  # NOISE, OBJECTIVE, SUBJECTIVE
        self.classifier2 = nn.Linear(hidden_size, 3)  # NEUTRAL, NEGATIVE, POSITIVE (within SUBJECTIVE)
        self.classifier3 = nn.Linear(hidden_size, 4)  # NEUTRAL SENTIMENTS, QUESTIONS, ADVERTISEMENTS, MISCELLANEOUS (within NEUTRAL)

    def forward(self, input_ids, attention_mask, level1_labels=None, level2_labels=None, level3_labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(outputs.last_hidden_state[:, 0])

        logits1 = self.classifier1(pooled)
        logits2 = self.classifier2(pooled)
        logits3 = self.classifier3(pooled)

        # For inference, we only return the logits
        return None, logits1, logits2, logits3


# --- Inference Function ---
@torch.no_grad() # Crucial for inference: disables gradient calculation
def predict_on_new_data(model, dataloader, device='cuda'):
    model.eval() # Set model to evaluation mode
    model.to(device)

    all_level1_preds = []
    all_level2_preds = []
    all_level3_preds = []

    print("\n🔮 Starting predictions on new data...")
    for batch in tqdm(dataloader, desc="Predicting"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        _, logits1, logits2, logits3 = model(input_ids, attention_mask)

        # Level 1 Prediction
        preds1 = torch.argmax(logits1, dim=1).cpu().tolist()
        all_level1_preds.extend(preds1)

        # Initialize Level 2 and Level 3 predictions as None/NaN or a placeholder
        # We'll fill them conditionally later
        current_batch_l2_preds = []
        current_batch_l3_preds = []

        # Process Level 2 and Level 3 predictions based on Level 1
        for i, l1_pred in enumerate(preds1):
            if l1_pred == 2:  # If Level 1 is SUBJECTIVE
                l2_pred = torch.argmax(logits2[i:i+1], dim=1).item()
                current_batch_l2_preds.append(l2_pred)
                
                if l2_pred == 0:  # If Level 2 is NEUTRAL
                    l3_pred = torch.argmax(logits3[i:i+1], dim=1).item()
                    current_batch_l3_preds.append(l3_pred)
                else:
                    current_batch_l3_preds.append(None) # Not applicable
            else: # If Level 1 is NOISE (0) or OBJECTIVE (1)
                current_batch_l2_preds.append(None) # Not applicable
                current_batch_l3_preds.append(None) # Not applicable
        
        all_level2_preds.extend(current_batch_l2_preds)
        all_level3_preds.extend(current_batch_l3_preds)

    return all_level1_preds, all_level2_preds, all_level3_preds


# --- Main Inference Execution ---
if __name__ == "__main__":
    # Ensure the model and tokenizer objects from your training run are still active.
    # If not, you would need to load the model state dict here:
    # MODEL_NAME = "SarkerLab/SocBERT-base"
    # tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    # model = HierarchicalClassifier(model_name=MODEL_NAME)
    # # Load the state_dict from your best saved model:
    # model_path_safetensors = os.path.join("saved_models", "best_reddit_tc_bert_hierarchical.safetensors")
    # if os.path.exists(model_path_safetensors):
    #     from safetensors import safe_open
    #     state_dict = {}
    #     with safe_open(model_path_safetensors, framework="pt", device="cpu") as f:
    #         for k in f.keys():
    #             state_dict[k] = f.get_tensor(k)
    #     model.load_state_dict(state_dict)
    #     print("Model loaded successfully for inference!")
    # else:
    #     print(f"Warning: Best model not found at {model_path_safetensors}. Ensure it was saved correctly or load another checkpoint.")
    #     # Exit or handle the error appropriately if the model cannot be loaded

    # Assume `model` and `tokenizer` are already loaded/trained from the previous script execution
    # and `device` is correctly set.

    if 'model' not in locals() or 'tokenizer' not in locals():
        print("Model or Tokenizer not found in the current environment. Please ensure the training script ran successfully or explicitly load them.")
        # As a fallback, you could add the loading logic here similar to the commented block above
        # for a fresh start or if the kernel reset.
        exit() # Exit if model/tokenizer are not available

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Load the new data for inference
    test_file_path = "/kaggle/input/train-task1/CRYPTO_YOUTUBE_TEST.csv"
    if not os.path.exists(test_file_path):
        print(f"Error: Test file not found at {test_file_path}. Please check the path.")
        exit()

    inference_df = pd.read_csv(test_file_path)

    # Create the inference dataset and dataloader
    inference_dataset = InferenceDataset(inference_df, tokenizer, max_len=128)
    inference_loader = DataLoader(inference_dataset, batch_size=16, shuffle=False)

    # Perform predictions
    level1_predictions, level2_predictions, level3_predictions = predict_on_new_data(model, inference_loader, device)

    # Add predictions to the DataFrame
    inference_df['level 1'] = level1_predictions
    inference_df['level 2'] = level2_predictions
    inference_df['level 3'] = level3_predictions
    
    # Handle NaN for levels where prediction is not applicable
    inference_df['level 2'] = inference_df['level 2'].fillna('')
    inference_df['level 3'] = inference_df['level 3'].fillna('')

    # Save the output file
    output_file_name = "reddit-tc-bert-crypto_test_youtube.csv"
    inference_df.to_csv(output_file_name, index=False)
    
    print(f"\nInference complete! Predictions saved to '{output_file_name}'")
    print("First 5 rows of the output file:")
    print(inference_df.head())